# Author's Notes

This `.ipynb` file is dedicated to reading the `.xml` (`XML` file format) file which contains information about each galaxy's identification number, redshift, etc. To make this file easier to handle, I will convert this file into a `.hdf5` file (`HDF5` file format).

In [11]:
import pandas as pd 
import xml.etree.ElementTree as ET

# Converting XML to .h5

## Reading XML

In [12]:
xml_location = "..\\dataset\\F02_UDEEP\\cesam_vvds_spF02_UDEEP_FULL.xml"
tree = ET.parse(xml_location)
root = tree.getroot()

In [13]:
for sub_root in root:
    print(sub_root)

<Element '{http://www.ivoa.net/xml/VOTable/v1.1}DESCRIPTION' at 0x000001D059891530>
<Element '{http://www.ivoa.net/xml/VOTable/v1.1}DEFINITIONS' at 0x000001D059891580>
<Element '{http://www.ivoa.net/xml/VOTable/v1.1}RESOURCE' at 0x000001D0598916C0>


In [14]:
namespace = {"v": "http://www.ivoa.net/xml/VOTable/v1.1"} # "v" can be anything. It is used to identify namespace
fields = root.findall("v:RESOURCE/v:TABLE/v:FIELD", namespace)

field_interest = ["NUM", "ID_IAU", "ALPHA", "DELTA", "MAGI", "Z", "ZFLAGS", "stellar_mass", "sfr"]
field_interest_index = []

for i in range(len(fields)):
    fieldname = fields[i].get("name")
    if fieldname in field_interest:
        field_interest_index.append(i)

print(field_interest_index)

[0, 1, 2, 3, 4, 5, 6, 201, 202]


In [ ]:
# For some unknown reason, the actual columns of the stellar_mass and sfr are 173 and 174, not 201 and 202, respectively.
field_interest_index = [0, 1, 2, 3, 4, 5, 6, 173, 174]

## Extraction of data from XML

In [16]:
tabledata_location = "v:RESOURCE/v:TABLE/v:DATA/v:TABLEDATA"
tabledata_row = root.findall(tabledata_location + "/v:TR", namespace)

In [17]:
dataframe = pd.DataFrame(columns = field_interest)

for row in tabledata_row:
    df_row = []

    for index in field_interest_index:
        data = row[index].text 
        df_row.append(data)
    dataframe.loc[len(dataframe)] = df_row

In [18]:
dataframe["NUM"] = dataframe["NUM"].astype(int)
dataframe["ID_IAU"] = dataframe["ID_IAU"].astype(str)
dataframe["ALPHA"] = dataframe["ALPHA"].astype(float)
dataframe["DELTA"] = dataframe["DELTA"].astype(float)
dataframe["MAGI"] = dataframe["MAGI"].astype(float)
dataframe["Z"] = dataframe["Z"].astype(float)
dataframe["ZFLAGS"] = dataframe["ZFLAGS"].astype(int)
dataframe["stellar_mass"] = dataframe["stellar_mass"].astype(float)
dataframe["sfr"] = dataframe["sfr"].astype(float)

In [19]:
dataframe.to_hdf("UltraDeep.h5", key = "population", mode = "a")